
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



# Batch Inference Using SLM

In this example, we will walk through some key steps for implementing an LLM-based pipeline using a **Small Language Model (SLM)** for batch inference in a production environment.

**Notes about this workflow:**

**📌 This notebook vs. modular scripts**: Since this demo is contained within a single notebook, we will divide the workflow from development to production into notebook sections. In a more realistic LLM Ops setup, these sections would likely be split into separate notebooks or scripts.

**📌 Promoting models vs. code**: We track the path from development to production via the Model Registry. That is, we are *promoting models* towards production, rather than promoting code.

## Learning Objectives

By the end of this demo, you should be able to:

1. Load a model from the Model Registry for batch inference.

1. Manage model aliases and retrieve the latest version of the model.

1. Apply batch inference on a Spark DataFrame using single-node batch inference.

1. Apply multinode batch inference using `spark_udf`.

1. Explain other methods of batch inference.

## REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:
1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

   - Click **More** in the drop-down.
   
   - In the **Attach to an existing compute resource** window, use the first drop-down to select your unique cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

2. Find the triangle icon to the right of your compute cluster name and click it.

3. Wait a few minutes for the cluster to start.

4. Once the cluster is running, complete the steps above to select your cluster.

## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need to use one of the following Databricks runtime(s): **15.4.x-cpu-ml-scala2.12**

## Classroom Setup

Install required libraries.

In [0]:
%pip install -qq -U huggingface-hub datasets transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
petastorm 0.12.1 requires pyspark>=2.1.0, which is not installed.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Before starting the demo, run the provided classroom setup script.

In [0]:
%pip install mlflow>=3.0 databricks-feature-engineering --upgrade
dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-server 1.23.4 requires anyio<4,>=3.1.0, but you have anyio 4.11.0 which is incompatible.
langchain 0.1.20 requires langchain-core<0.2.0,>=0.1.52, but you have langchain-core 1.0.5 which is incompatible.
langchain 0.1.20 requires langsmith<0.2.0,>=0.1.17, but you have langsmith 0.4.43 which is incompatible.
langchain 0.1.20 requires tenacity<9.0.0,>=8.1.0, but you have tenacity 9.1.2 which is incompatible.
langchain-community 0.0.38 requires langchain-core<0.2.0,>=0.1.52, but you have langchain-core 1.0.5 which is incompatible.
langchain-community 0.0.38 requires langsmith<0.2.0,>=0.1.0, but you have langsmith 0.4.43 which is incompatible.
langchain-community 0.0.38 requires tenacity<9.0.0,>=8.1.0, but you have tenacity 9.1.2 which is incompatible.
langchain-text-splitters 0.0.2 requires langchain-core<0.3,

In [0]:
%run ../Includes/Classroom-Setup-01

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
databricks-agents 1.8.2 requires databricks-sdk[openai]>=0.58.0, but you have databricks-sdk 0.36.0 which is incompatible.
databricks-feature-engineering 0.13.0 requires databricks-sdk>=0.62.0, but you have databricks-sdk 0.36.0 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.



The examples and models presented in this course are intended solely for demonstration and educational purposes.
 Please note that the models and prompt examples may sometimes contain offensive, inaccurate, biased, or harmful content.


**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")

Username:          labuser12678309_1763243239@vocareum.com
Catalog Name:      dbacademy
Schema Name:       labuser12678309_1763243239
Working Directory: /Volumes/dbacademy/ops/labuser12678309_1763243239@vocareum_com
Dataset Location:  NestedNamespace (arxiv='/Volumes/dbacademy_arxiv/v01')


## Demo Overview

1. Prepare dataset.
1. Develop a Huggingface/transformer LLM pipeline.
1. Apply/test pipeline to data, and log results to MLflow Tracking.
1. Log the pipeline to the MLflow Tracking server as an MLflow Model.
1. Load LLM pipeline from registry and run batch inference
1. Use SQL `ai_query()` for batch inference on existing/supported _Foundation Models API_ models

## Data and Model Preparation 

In this section, we will create a dataset and the model that we will be using for the rest of the demo.

### Prepare Dataset

Prepare a Delta table containing texts to summarize from the [Extreme Summarization (XSum) Dataset](https://huggingface.co/datasets/EdinburghNLP/xsum), which we will use to run batch inferences.

In [0]:
# This is how to use dataset EdinburghNLP/xsum (https://huggingface.co/datasets/EdinburghNLP/xsum) from HuggginFace
from datasets import load_dataset

# hf_imdb = load_dataset("imdb")
hf_edin = load_dataset(
    "EdinburghNLP/xsum", 
    revision="refs/convert/parquet"
)

train_hf_edin_df = spark.createDataFrame(
    hf_edin["train"].to_pandas()
)
validation_hf_edin_df = spark.createDataFrame(
    hf_edin["validation"].to_pandas()
)
test_hf_edin_df = spark.createDataFrame(
    hf_edin["test"].to_pandas()
)

display(train_hf_edin_df.limit(20))
display(validation_hf_edin_df.limit(20))
display(test_hf_edin_df.limit(20))

/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:45: UserWarning: The cache_dir for this dataset is /root/.cache, which is not a persistent path.Therefore, if/when the cluster restarts, the downloaded dataset will be lost.The persistent storage options for this workspace/cluster config are: [DBFS, UC Volumes].Please update either `cache_dir` or the environment variable `HF_DATASETS_CACHE`to be under one of the following root directories: ['/dbfs/', '/Volumes/']
  warnings.warn(warning_message)


document summary id The full cost of damage in Newton Stewart, one of the areas worst affected, is still being assessed.
Repair work is ongoing in Hawick and many roads in Peeblesshire remain badly affected by standing water.
Trains on the west coast mainline face disruption due to damage at the Lamington Viaduct.
Many businesses and householders were affected by flooding in Newton Stewart after the River Cree overflowed into the town.
First Minister Nicola Sturgeon visited the area to inspect the damage.
The waters breached a retaining wall, flooding many commercial properties on Victoria Street - the main shopping thoroughfare.
Jeanette Tate, who owns the Cinnamon Cafe which was badly affected, said she could not fault the multi-agency response once the flood hit.
However, she said more preventative work could have been carried out to ensure the retaining wall did not fail.
"It is difficult but I do think there is so much publicity for Dumfries and the Nith - and I totally appreciate that - but it is almost like we're neglected or forgotten," she said.
"That may not be true but it is perhaps my perspective over the last few days.
"Why were you not ready to help us a bit more when the warning and the alarm alerts had gone out?"
Meanwhile, a flood alert remains in place across the Borders because of the constant rain.
Peebles was badly hit by problems, sparking calls to introduce more defences in the area.
Scottish Borders Council has put a list on its website of the roads worst affected and drivers have been urged not to ignore closure signs.
The Labour Party's deputy Scottish leader Alex Rowley was in Hawick on Monday to see the situation first hand.
He said it was important to get the flood protection plan right but backed calls to speed up the process.
"I was quite taken aback by the amount of damage that has been done," he said.
"Obviously it is heart-breaking for people who have been forced out of their homes and the impact on businesses."
He said it was important that "immediate steps" were taken to protect the areas most vulnerable and a clear timetable put in place for flood prevention plans.
Have you been affected by flooding in Dumfries and Galloway or the Borders? Tell us about your experience of the situation and how it was handled. Email us on selkirk.news@bbc.co.uk or dumfries@bbc.co.uk. Clean-up operations are continuing across the Scottish Borders and Dumfries and Galloway after flooding caused by Storm Frank. 35232142 A fire alarm went off at the Holiday Inn in Hope Street at about 04:20 BST on Saturday and guests were asked to leave the hotel.
As they gathered outside they saw the two buses, parked side-by-side in the car park, engulfed by flames.
One of the tour groups is from Germany, the other from China and Taiwan. It was their first night in Northern Ireland.
The driver of one of the buses said many of the passengers had left personal belongings on board and these had been destroyed.
Both groups have organised replacement coaches and will begin their tour of the north coast later than they had planned.
Police have appealed for information about the attack.
Insp David Gibson said: "It appears as though the fire started under one of the buses before spreading to the second.
"While the exact cause is still under investigation, it is thought that the fire was started deliberately." Two tourist buses have been destroyed by fire in a suspected arson attack in Belfast city centre. 40143035 Ferrari appeared in a position to challenge until the final laps, when the Mercedes stretched their legs to go half a second clear of the red cars.
Sebastian Vettel will start third ahead of team-mate Kimi Raikkonen.
The world champion subsequently escaped punishment for reversing in the pit lane, which could have seen him stripped of pole.
But stewards only handed Hamilton a reprimand, after governing body the FIA said "no clear instruction was given on where he should park".
Belgian Stoffel Vandoorne out-qualified McLaren 

document summary id The ex-Reading defender denied fraudulent trading charges relating to the Sodje Sports Foundation - a charity to raise money for Nigerian sport.
Mr Sodje, 37, is jointly charged with elder brothers Efe, 44, Bright, 50 and Stephen, 42.
Appearing at the Old Bailey earlier, all four denied the offence.
The charge relates to offences which allegedly took place between 2008 and 2014.
Sam, from Kent, Efe and Bright, of Greater Manchester, and Stephen, from Bexley, are due to stand trial in July.
They were all released on bail. Former Premier League footballer Sam Sodje has appeared in court alongside three brothers accused of charity fraud. 38295789 Voges was forced to retire hurt on 86 after suffering the injury while batting during the County Championship draw with Somerset on 4 June.
Middlesex hope to have the Australian back for their T20 Blast game against Hampshire at Lord's on 3 August.
The 37-year-old has scored 230 runs in four first-class games this season at an average of 57.50.
"Losing Adam is naturally a blow as he contributes significantly to everything we do," director of cricket Angus Fraser said.
"His absence, however, does give opportunities to other players who are desperate to play in the first XI.
"In the past we have coped well without an overseas player and I expect us to do so now."
Defending county champions Middlesex are sixth in the Division One table, having drawn all four of their matches this season.
Voges retired from international cricket in February with a Test batting average of 61.87 from 31 innings, second only to Australian great Sir Donald Bradman's career average of 99.94 from 52 Tests. Middlesex batsman Adam Voges will be out until August after suffering a torn calf muscle in his right leg. 40202028 Seven photographs taken in the Norfolk countryside by photographer Josh Olins will appear in the June edition.
In her first sitting for a magazine, the duchess is seen looking relaxed and wearing casual clothes.
The shoot was in collaboration with the National Portrait Gallery, where two images are being displayed in the Vogue 100: A Century of Style exhibition.
The duchess, who has a keen interest in photography, has been patron of the National Portrait Gallery since 2012.
Nicholas Cullinan, director of the National Portrait Gallery, said: "Josh has captured the duchess exactly as she is - full of life, with a great sense of humour, thoughtful and intelligent, and in fact, very beautiful."
He said the images also encapsulated what Vogue had done over the past 100 years - "to pair the best photographers with the great personalities of the day, in order to reflect broader shifts in culture and society".
Alexandra Shulman, editor-in-chief of British Vogue, said: "To be able to publish a photographic shoot with the Duchess of Cambridge has been one of my greatest ambitions for the magazine."
The collaboration for the June edition had resulted in "a true celebration of our centenary as well as a fitting tribute to a young woman whose interest in both photography and the countryside is well known", she said.
Other royal portraits to have featured in the fashion magazine include Diana, Princess of Wales - who graced the cover four times - and Princess Anne.
The duchess is to visit the exhibition at the National Portrait Gallery on Wednesday, Kensington Palace said. The Duchess of Cambridge will feature on the cover of British Vogue to mark the magazine's centenary. 36177725 Chris Poole - known as "moot" online - created the site in 2003.
It has gone on to be closely associated with offensive and often illegal activity, including instances where the images of child abuse were shared.
It was widely credited as being the first place where leaked images of nude celebrities were posted following 2014's well-publicised security breach affecting Apple's iCloud service. That incident prompted a policy change on the site.
However, 4chan has also been the rallying point for many instances of on

document summary id Prison Link Cymru had 1,099 referrals in 2015-16 and said some ex-offenders were living rough for up to a year before finding suitable accommodation.
Workers at the charity claim investment in housing would be cheaper than jailing homeless repeat offenders.
The Welsh Government said more people than ever were getting help to address housing problems.
Changes to the Housing Act in Wales, introduced in 2015, removed the right for prison leavers to be given priority for accommodation.
Prison Link Cymru, which helps people find accommodation after their release, said things were generally good for women because issues such as children or domestic violence were now considered.
However, the same could not be said for men, the charity said, because issues which often affect them, such as post traumatic stress disorder or drug dependency, were often viewed as less of a priority.
Andrew Stevens, who works in Welsh prisons trying to secure housing for prison leavers, said the need for accommodation was "chronic".
"There's a desperate need for it, finding suitable accommodation for those leaving prison there is just a lack of it everywhere," he said.
"It could take six months to a year, without a lot of help they could be on the streets for six months.
"When you think of the consequences of either being on the street, especially with the cold weather at the moment or you may have a roof over your head, sometimes there is only one choice."
Mr Stevens believes building more one-bedroom flats could help ease the problem.
"The average price is a hundred pounds a week to keep someone in a rented flat, prison is a lot more than that so I would imagine it would save the public purse quite a few pounds," he said.
Official figures show 830 one-bedroom properties were built in the year to March 2016, of an overall total of 6,900 new properties in Wales.
Marc, 50, who has been in and out of prison for the past 20 years for burglary offences, said he struggled to find accommodation each time he was released.
He said he would ask himself: "Where am I going to stay? Where am I going to live? Have I got somewhere where I can see my daughter."
"You're put out among the same sort of people doing the same sort of thing, and it's difficult, it's difficult to get away from it. It's like every man for himself, there's nothing."
Marc has now found stable accommodation with homeless charity Emmaus and said it had been life changing.
"You feel safe, you got hot food, you've got company of people in similar situations to yourself but all dealing with different issues. It's a constructive, helpful atmosphere," he said.
Tom Clarke, chief executive of Emmaus South Wales, agreed there was not enough support available.
"We do still see [people] homeless on the streets, so clearly they haven't got accommodation and haven't got provision," he said.
"I think the key is connecting people with the services they need. I don't delude myself that Emmaus can offer a one size fits all for everyone, we can't.
"But there must be other opportunities and given suitable encouragement I believe that can and should happen."
A Welsh Government spokesman said the national pathway for homeless services to children, young people and adults in the secure estate had prevented many people from losing their home whilst serving their prison sentence.
It added there were already significant demands for one-bedroom flats across the public and private sector and it was providing 20,000 new affordable homes in the next five years. There is a "chronic" need for more housing for prison leavers in Wales, according to a charity. 38264402 Officers searched properties in the Waterfront Park and Colonsay View areas of the city on Wednesday.
Detectives said three firearms, ammunition and a five-figure sum of money were recovered.
A 26-year-old man who was arrested and charged appeared at Edinburgh Sheriff Court on Thursday. A man has appeared in court after firearms, ammunition and ca

In [0]:
dbutils.fs.ls(DA.paths.working_dir)

[FileInfo(path='dbfs:/Volumes/dbacademy/ops/labuser12678309_1763243239@vocareum_com/hf_cache/', name='hf_cache/', size=0, modificationTime=1763246115952)]

In [0]:
from pyspark.sql.functions import input_file_name, col, from_utc_timestamp

cache_dir = DA.paths.working_dir + "/hf_cache"
files_df = spark.createDataFrame(dbutils.fs.ls(cache_dir))
files_df = files_df.withColumnRenamed("name", "filename") \
    .withColumnRenamed("modificationTime", "modification_utc") \
    .withColumnRenamed("path", "path") \
    .withColumn("modification_chicago", from_utc_timestamp((col("modification_utc")/1000).cast("timestamp"), "America/Chicago")) \
    .select("filename", "modification_chicago", "path")

display(files_df)

filename,modification_chicago,path
EdinburghNLP___xsum/,2025-11-15T16:37:03.358Z,dbfs:/Volumes/dbacademy/ops/labuser12678309_1763243239@vocareum_com/hf_cache/EdinburghNLP___xsum/
_Volumes_dbacademy_ops_labuser12678309_1763243239@vocareum_com_hf_cache_EdinburghNLP___xsum_default_0.0.0_b46d1408a83c7c650e4e3605e24dad5c9e06297a.lock,2025-11-15T16:12:40Z,dbfs:/Volumes/dbacademy/ops/labuser12678309_1763243239@vocareum_com/hf_cache/_Volumes_dbacademy_ops_labuser12678309_1763243239@vocareum_com_hf_cache_EdinburghNLP___xsum_default_0.0.0_b46d1408a83c7c650e4e3605e24dad5c9e06297a.lock
_local_disk0_hf_cache_EdinburghNLP___xsum_default_0.0.0_b46d1408a83c7c650e4e3605e24dad5c9e06297a.lock,2025-11-15T16:16:08Z,dbfs:/Volumes/dbacademy/ops/labuser12678309_1763243239@vocareum_com/hf_cache/_local_disk0_hf_cache_EdinburghNLP___xsum_default_0.0.0_b46d1408a83c7c650e4e3605e24dad5c9e06297a.lock


In [0]:
import datasets
from datasets import load_dataset
from transformers import pipeline
from delta.tables import DeltaTable
prod_data_table_name = f"{DA.catalog_name}.{DA.schema_name}.m4_1_prod_data"
datasets.utils.logging.disable_progress_bar()
xsum_dataset = load_dataset(
    "EdinburghNLP/xsum",
    revision="refs/convert/parquet",
    cache_dir=DA.paths.working_dir + "/hf_cache"
)
# Save test set to delta table
test_spark_df = spark.createDataFrame(xsum_dataset["test"].to_pandas())
test_spark_df.write.mode("overwrite").saveAsTable(prod_data_table_name)

2025-11-15 22:15:45.540061: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-15 22:15:45.544576: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-15 22:15:45.593464: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-15 22:15:46.687582: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:127: UserWarning: The dataset would be saved to both local disk and PersistentStorageType.VOLUMES for better performance.
  warnings.warn(


default/train/0000.parquet:   0%|          | 0.00/304M [00:00<?, ?B/s]

default/validation/0000.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

default/test/0000.parquet:   0%|          | 0.00/17.0M [00:00<?, ?B/s]

### Create a Hugging Face Pipeline

##### 1. For this notebook we'll use the <a href="https://huggingface.co/t5-small" target="_blank">T5 Text-To-Text Transfer Transformer</a> from Hugging Face.



**T5-Small model (60M parameters) :  is used for machine translation, document summarization, question answering, and classification tasks (like sentiment analysis). It can even handle regression tasks by training it to predict the string representation of a number.**



##### 2. For this notebook we'll use the <a href="https://huggingface.co/docs/transformers/v4.36.1/main_classes/pipelines" target="_blank">Hugging Face Pipeline</a> to use models for inference. 


| Task                                 | Returned Pipeline                       |
|-------------------------------------- |-----------------------------------------|
| audio-classification                  | AudioClassificationPipeline             |
| automatic-speech-recognition          | AutomaticSpeechRecognitionPipeline      |
| conversational                        | ConversationalPipeline                  |
| depth-estimation                      | DepthEstimationPipeline                 |
| document-question-answering           | DocumentQuestionAnsweringPipeline       |
| feature-extraction                    | FeatureExtractionPipeline               |
| fill-mask                             | FillMaskPipeline                        |
| image-classification                  | ImageClassificationPipeline             |
| image-segmentation                    | ImageSegmentationPipeline               |
| image-to-image                        | ImageToImagePipeline                    |
| image-to-text                         | ImageToTextPipeline                     |
| mask-generation                       | MaskGenerationPipeline                  |
| object-detection                      | ObjectDetectionPipeline                 |
| question-answering                    | QuestionAnsweringPipeline               |
| summarization                         | SummarizationPipeline                   |
| table-question-answering              | TableQuestionAnsweringPipeline          |
| text2text-generation                  | Text2TextGenerationPipeline             |
| text-classification / sentiment-analysis | TextClassificationPipeline           |
| text-generation                       | TextGenerationPipeline                  |
| text-to-audio / text-to-speech        | TextToAudioPipeline                     |
| token-classification / ner            | TokenClassificationPipeline             |
| translation                           | TranslationPipeline                     |
| translation_xx_to_yy                  | TranslationPipeline                     |
| video-classification                  | VideoClassificationPipeline             |
| visual-question-answering             | VisualQuestionAnsweringPipeline         |
| zero-shot-classification              | ZeroShotClassificationPipeline          |
| zero-shot-image-classification        | ZeroShotImageClassificationPipeline     |
| zero-shot-audio-classification        | ZeroShotAudioClassificationPipeline     |
| zero-shot-object-detection            | ZeroShotObjectDetectionPipeline         |


In [0]:
from transformers import pipeline

# Define pipeline inference parameters - to be logged in mlflow as part of model _metadata
hf_model_name = "t5-small" 
min_length = 20
max_length = 40
truncation = True
do_sample = True
device_map = "auto" # 'cuda', 'cpu'

cache_dir = "/hf_cache" 

summarizer = pipeline(
    task="summarization", # define task for model in Hugging Face pipeline
    model=hf_model_name,  # declare model that will be used in Hugging Face pipeline
    min_length=min_length,
    max_length=max_length,
    truncation=truncation,
    do_sample=do_sample,
    device_map=device_map,
    model_kwargs={"cache_dir": cache_dir},
)  # Note: We specify cache_dir to use pre-cached models.

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Device set to use cpu


In [0]:
# Get all attributes from SummarizationPipeline `summarizer`
dir(summarizer)

['__abstractmethods__',
 '__annotations__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_batch_size',
 '_create_repo',
 '_default_generation_config',
 '_ensure_tensor_on_device',
 '_forward',
 '_forward_params',
 '_get_files_timestamps',
 '_load_feature_extractor',
 '_load_image_processor',
 '_load_processor',
 '_load_tokenizer',
 '_num_workers',
 '_parse_and_tokenize',
 '_pipeline_calls_generate',
 '_postprocess_params',
 '_preprocess_params',
 '_sanitize_parameters',
 '_upload_modified_files',
 'assistant_model',
 'assistant_tokenizer',
 'binary_output',
 'call_count',
 'check_inputs',
 'check_model_type'

We can now examine the `summarizer` pipeline summarizing some text

In [0]:
text_to_summarize= """ Barrington DeVaughn Hendricks (born October 22, 1989), known professionally as JPEGMafia (stylized in all caps), is an American rapper, singer, and record producer born in New York City and based in Baltimore, Maryland. His 2018 album Veteran, released through Deathbomb Arc, received widespread critical acclaim and was featured on many year-end lists. It was followed by 2019's All My Heroes Are Cornballs and 2021's LP!, released to further critical acclaim. """

summarized_text = summarizer(text_to_summarize)[0]["summary_text"]
print(f"Summary:\n {summarized_text}")
print("===============================================")
print(f"Original Document: {text_to_summarize}")

Your max_length is set to 200, but your input_length is only 124. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=62)


Summary:
 Barrington DeVaughn Hendricks (born October 22, 1989) is an american rapper, singer, record producer . his 2018 album Veteran, released through Deathbomb Arc, received widespread critical acclaim .
Original Document:  Barrington DeVaughn Hendricks (born October 22, 1989), known professionally as JPEGMafia (stylized in all caps), is an American rapper, singer, and record producer born in New York City and based in Baltimore, Maryland. His 2018 album Veteran, released through Deathbomb Arc, received widespread critical acclaim and was featured on many year-end lists. It was followed by 2019's All My Heroes Are Cornballs and 2021's LP!, released to further critical acclaim. 


In [0]:
text_to_summarize_2= """ A State Department spokesman said the election process was flawed and could not be seen as free and fair. He said the Ortega government had side-lined opposition candidates and limited monitoring of the polls. Daniel Ortega won 72.5% of the vote with 99.8% of the ballots counted. His closest rival, centre-right candidate Maximino Rodriguez, only received 14.2% of the vote. The State Department's Mark Toner said the Ortega government had not invited international election observers, which he said, "further degraded the legitimacy of the election". "We continue to press the Nicaraguan government to uphold democratic practices, including press freedom and respect for universal human rights in Nicaragua," he added. Mr Ortega had been widely expected to win both due to the popularity of his social programmes and because he faced no obvious political challenger. A former left-wing rebel, Mr Ortega has led Nicaragua through a period of economic stability which has made him popular with the country's business sector and foreign investors. Supporters of Mr Ortega took to the streets to celebrate his victory. But even before the first results were announced, members of the opposition coalition Broad Front for Democracy (FAD) called the elections a "farce". The FAD, which had urged voters to boycott the election, alleged that more than 70% had abstained from voting. They were contradicted by the electoral authorities which put voter participation at 65.8%. Mr Ortega's running mate was his wife, Rosario Murillo, who now looks set to become vice-president. Analysts say that Ms Murillo already shares decision-making with Mr Ortega and could become president if her 70-year-old husband were to bow out. Nicaragua's economy has grown at double the Latin American average, but the country still needs to attract more foreign investment. A $50bn (£40bn) plan to build an interoceanic canal across Nicaragua with Chinese investment has gained international attention, but there are serious doubts over whether it will ever be built. The country has been able to avoid the sky-high murder rates of some of its Central American neighbours but it also faces the ever pervasive threat of drug-trafficking. """

summarized_text_2 = summarizer(text_to_summarize_2)[0]["summary_text"]
print(f"Summary:\n {summarized_text_2}")
print("===============================================")
print(f"Original Document: {text_to_summarize_2}")

Summary:
 a state department spokesman says the election process is flawed . the government has side-lined opposition candidates and limited monitoring of the polls . his closest rival, centre-right candidate Maximino Rodriguez, only received 14.2% of the vote .
Original Document:  A State Department spokesman said the election process was flawed and could not be seen as free and fair. He said the Ortega government had side-lined opposition candidates and limited monitoring of the polls. Daniel Ortega won 72.5% of the vote with 99.8% of the ballots counted. His closest rival, centre-right candidate Maximino Rodriguez, only received 14.2% of the vote. The State Department's Mark Toner said the Ortega government had not invited international election observers, which he said, "further degraded the legitimacy of the election". "We continue to press the Nicaraguan government to uphold democratic practices, including press freedom and respect for universal human rights in Nicaragua," he add

## Model Development and Registering

### Track LLM Development with MLflow

Before we start the model development, here is a quick refresher for MLflow tracking.

[MLflow](https://mlflow.org/) Tracking helps track model or pipeline production during development. Even without fitting a model, you can use it to track example queries and responses to the LLM pipeline and store the model as an [MLflow Model flavor](https://mlflow.org/docs/latest/models.html#built-in-model-flavors) for simpler deployment. 

MLflow Tracking is organized hierarchically: an [experiment](https://mlflow.org/docs/latest/tracking.html#organizing-runs-in-experiments) corresponds to creating a primary model or pipeline, containing multiple [runs](https://mlflow.org/docs/latest/tracking.html#organizing-runs-in-experiments). Each run logs parameters, metrics, tags, models, artifacts, and other metadata. Parameters are inputs like `max_length`, metrics are evaluation outputs like accuracy, and artifacts are files like the serialized model. A [flavor](https://mlflow.org/docs/latest/models.html#storage-format) is an MLflow format for serializing models using the underlying ML library's format plus metadata. For more information, see the [LLM Tracking page](https://mlflow.org/docs/latest/llms/llm-tracking/index.html). Tip: Wrap your model development workflow with `with mlflow.start_run():` to start and end the MLflow run explicitly, a best practice for production code. See the [API doc](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.start_run) for more details.

In [0]:
import mlflow
from mlflow.models import infer_signature
from mlflow.transformers import generate_signature_output

# It is valuable to log a "signature" with the model telling MLflow the input and output schema for the model.
output = generate_signature_output(summarizer, text_to_summarize)
signature = infer_signature(text_to_summarize, output)
print(f"output:\n{output}\n")
print(f"Signature:\n{signature}\n")

Your max_length is set to 200, but your input_length is only 124. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=62)


output:
['Barrington deVaughn Hendricks (born October 22, 1989) is an american rapper, singer, and record producer . his 2018 album Veteran, released through Deathbomb Arc, received widespread critical acclaim .']

Signature:
inputs: 
  [string (required)]
outputs: 
  [string (required)]
params: 
  None




In [0]:
import mlflow
from mlflow.models import infer_signature
from mlflow.transformers import generate_signature_output


# It is valuable to log a "signature" with the model telling MLflow the input and output schema for the model.
output = generate_signature_output(summarizer, text_to_summarize)
signature = infer_signature(text_to_summarize, output)
print(f"Signature:\n{signature}\n")


# Set experiment path
# (located on the left hand sidebar under Machine Learning -> Experiments)
model_artifact_path = "summarizer"
experiment_name = f"/Users/{DA.username}/GenAI-As-04-Batch-Demo"
mlflow.set_experiment(experiment_name)
model_artifact_path = "summarizer" # Name of folder containing serialized model

with mlflow.start_run():
    # LOG PARAMS
    mlflow.log_params(
        {
            "hf_model_name": hf_model_name,
            "min_length": min_length,
            "max_length": max_length,
            "truncation": truncation,
            "do_sample": do_sample,
        }
    )

    # ---------
    # LOG MODEL
    # We next log our LLM pipeline as an MLflow model.
    # This packages the model with useful metadata, such as the library versions used to create it.
    # This metadata makes it much easier to deploy the model downstream.
    # Under the hood, the model format is simply the ML library's native format (Hugging Face for us), plus metadata.

    # For mlflow.transformers, if there are inference-time configurations,
    # those need to be saved specially in the log_model call (below).
    # This ensures that the pipeline will use these same configurations when re-loaded.
    inference_config = {
        "min_length": min_length,
        "max_length": max_length,
        "truncation": truncation,
        "do_sample": do_sample,
    }

    # Logging a model returns a handle `model_info` to the model metadata in the tracking server.
    # This `model_info` will be useful later in the notebook to retrieve the logged model.
    model_info = mlflow.transformers.log_model(
        transformers_model=summarizer,
        artifact_path=model_artifact_path,
        task="summarization",
        inference_config=inference_config,
        signature=signature,
        input_example="This is an example of a long news article which this pipeline can summarize for you.",
    )

Signature:
inputs: 
  [string (required)]
outputs: 
  [string (required)]
params: 
  None




2025/11/15 22:49:54 INFO mlflow.tracking.fluent: Experiment with name '/Users/labuser12678309_1763243239@vocareum.com/GenAI-As-04-Batch-Demo' does not exist. Creating a new experiment.
2025/11/15 22:49:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[2025-11-15 22:49:56,335] [WARNING] [real_accelerator.py:162:get_accelerator] Setting accelerator to CPU. If you have GPU or other accelerator, we were unable to detect it.
[2025-11-15 22:49:56,341] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cpu (auto detect)


🔗 View Logged Model at: https://dbc-adde1550-8a14.cloud.databricks.com/ml/experiments/4117165248792101/models/m-89f313273fcd430da9666bad27d54e8c?o=253704115258622


README.md:   0%|          | 0.00/8.47k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cpu
Your max_length is set to 200, but your input_length is only 19. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=9)


### Query the MLflow Tracking server

**MLflow Tracking API**: We briefly show how to query the logged model and metadata in the MLflow Tracking server, by loading the logged model.  See the [MLflow API](https://mlflow.org/docs/latest/python_api/mlflow.html) for more information about programmatic access.

**MLflow Tracking UI**: You can also use the UI.  In the right-hand sidebar, click the [MLflow experiments](/ml/experiments) to view the run list, and then click through to access the Tracking server UI.  There, you can see the logged metadata and model.  Note in particular that our LLM inputs and outputs have been logged as a CSV file under model artifacts.

GIF of MLflow UI:

![llmops](../Includes/images/llmops.gif)


In [0]:
# Grab most recent run (which logged the model) using our experiment ID
experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
runs = mlflow.search_runs([experiment_id]) 
last_run_id = runs.sort_values("start_time", ascending=False).iloc[0].run_id # Experiment ID can have multiple Run IDs so we extract the latest run id based on start time

# Construct model URI based on run_id
model_uri = f"runs:/{last_run_id}/{model_artifact_path}"

In [0]:
model_uri

'runs:/14ea559047da4983938766dd8aaebae4/summarizer'

### Load Model Back as a Pipeline

Now, we can load the pipeline back from MLflow as a [pyfunc](https://mlflow.org/docs/latest/python_api/mlflow.pyfunc.html) and use the `.predict()` method to summarize an example document.

In [0]:
loaded_summarizer = mlflow.pyfunc.load_model(model_uri=model_uri) # mlflow.pyfunc is suitable fior Row-wise inference, while mlflow.pyfunc.spark_udf is used to perform high-throughput batch inference across a large, distributed Spark DataFrame. 
loaded_summarizer.predict(text_to_summarize)

Device set to use cpu
Your max_length is set to 200, but your input_length is only 122. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=61)


['Barrington DeVaughn Hendricks (born October 22, 1989) is an American rapper, singer, and record producer born in Baltimore, Maryland . Barrington deVaughs (stylized in all caps) is a former acclaimer of his 2018 album Veteran, released through Deathbomb Arc . he is credited with promoting his new album, Veteran .']

**Note :** The `.predict()` method can handle more than one document at a time (like a `pd.Series()` or `list()`)

### Register the Model to Unity Catalog

Register the pipeline to Unity-Catalog's Model Registry and set a model alias to label model as ready for staging/QA for example

We track our progress here using **Unity Catalog's Model Registry** [AWS](https://docs.databricks.com/en/machine-learning/manage-model-lifecycle/index.html) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/manage-model-lifecycle/) | [GCP](https://docs.gcp.databricks.com/en/machine-learning/manage-model-lifecycle/index.html). 
This metadata and model store organizes models as follows:
* **A registered model** is a named model in the registry (respecting 3-level namespace convention ***`catalog.schema.model_name`***), in our case corresponding to our summarization model.  It may have multiple *versions*.
   * **A model version** is an instance of a given model.  As you update your model, you will create new versions.  Each version could be designated as being in a particular stage of deployment via:
      * **An `@alias`** is a unique - free text- alias describing which stage of deployment (e.g. `challenger` (development), `champion` (production), `baseline` or `archived`).

The model we registered above starts with 1 version and no @alias.

In the workflow below, we will programmatically modify/set the `@alias` for given model versions in order to mark their stage.  For more information on the Model Registry API, see the [Model Registry docs](https://mlflow.org/docs/latest/model-registry.html).  Alternatively, you can edit the registry and set model @aliases via the UI.

In [0]:
from mlflow import MlflowClient

# Define model name in the Model Registry
model_name = f"{DA.catalog_name}.{DA.schema_name}.summarizer"

# Point to Unity-Catalog registry and log/push artifact
mlflow.set_registry_uri("databricks-uc")
mlflow.register_model(
    model_uri=model_uri,
    name=model_name,
)

Registered model 'dbacademy.labuser12678309_1763243239.summarizer' already exists. Creating a new version of this model...
2025/11/15 23:12:00 WARNING mlflow.tracking._model_registry.fluent: Run with id 14ea559047da4983938766dd8aaebae4 has no artifacts at artifact path 'summarizer', registering model based on models:/m-89f313273fcd430da9666bad27d54e8c instead


Uploading artifacts:   0%|          | 0/21 [00:00<?, ?it/s]

🔗 Created version '2' of model 'dbacademy.labuser12678309_1763243239.summarizer': https://dbc-adde1550-8a14.cloud.databricks.com/explore/data/models/dbacademy/labuser12678309_1763243239/summarizer/version/2?o=253704115258622


<ModelVersion: aliases=[], creation_timestamp=1763248328289, current_stage=None, deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1763248333786, metrics=[], model_id='m-89f313273fcd430da9666bad27d54e8c', name='dbacademy.labuser12678309_1763243239.summarizer', params=[<LoggedModelParameter: key='hf_model_name', value='t5-small'>,
 <LoggedModelParameter: key='do_sample', value='True'>,
 <LoggedModelParameter: key='truncation', value='True'>,
 <LoggedModelParameter: key='max_length', value='40'>,
 <LoggedModelParameter: key='min_length', value='20'>], run_id='14ea559047da4983938766dd8aaebae4', run_link=None, source='models:/m-89f313273fcd430da9666bad27d54e8c', status='READY', status_message='', tags={}, user_id='labuser12678309_1763243239@vocareum.com', version='2'>

## Manage Model Stage

Set latest model version as `@champion`

In [0]:
def get_latest_model_version(model_name_in):
    """
    Helper method to programmatically get latest model's version from the registry
    """
    client = MlflowClient()
    model_version_infos = client.search_model_versions("name = '%s'" % model_name_in)
    return max([model_version_info.version for model_version_info in model_version_infos])

In [0]:
# Set @alias
client = mlflow.tracking.MlflowClient()
current_model_version = get_latest_model_version(model_name)

client.set_registered_model_alias(
  name=model_name, alias="champion",
  version=current_model_version
  )

## Create a Production Workflow for Batch Inference

In production the goals are (a) to write scale-out code which can meet scaling demands in the future and (b) to simplify deployment by using MLflow to write model-agnostic deployment code.  Step-by-step, we will:
* Load the latest production LLM pipeline from the Model Registry.
* Apply the pipeline to an Apache Spark DataFrame.
* Append the results to a Delta Lake table.

Here, we will show batch inference using Apache Spark DataFrames, with Delta Lake format.  Spark allows simple scale-out inference for high-throughput, low-cost jobs, and Delta allows us to append to and modify inference result tables with ACID transactions.  See the [Apache Spark page](https://spark.apache.org/) and the [Delta Lake page](https://delta.io/) more information on these technologies.

*Model URIs*: Below, we use model URIs to tell MLflow which model and version we are referring to.  Two common URI patterns for the MLflow Model Registry are:
* `f"models:/{model_name}/{model_version}"` to refer to a specific model version by number
* `f"models:/{model_name}@{alias}"` to refer to the model version given unique @alias

Before we start, let's load the input texts to summarize into a spark dataframe

In [0]:
prod_data_table = f"{DA.catalog_name}.{DA.schema_name}.m4_1_prod_data"
prod_data_df = spark.read.table(prod_data_table).limit(10)
display(prod_data_df)

document summary id Prison Link Cymru had 1,099 referrals in 2015-16 and said some ex-offenders were living rough for up to a year before finding suitable accommodation.
Workers at the charity claim investment in housing would be cheaper than jailing homeless repeat offenders.
The Welsh Government said more people than ever were getting help to address housing problems.
Changes to the Housing Act in Wales, introduced in 2015, removed the right for prison leavers to be given priority for accommodation.
Prison Link Cymru, which helps people find accommodation after their release, said things were generally good for women because issues such as children or domestic violence were now considered.
However, the same could not be said for men, the charity said, because issues which often affect them, such as post traumatic stress disorder or drug dependency, were often viewed as less of a priority.
Andrew Stevens, who works in Welsh prisons trying to secure housing for prison leavers, said the need for accommodation was "chronic".
"There's a desperate need for it, finding suitable accommodation for those leaving prison there is just a lack of it everywhere," he said.
"It could take six months to a year, without a lot of help they could be on the streets for six months.
"When you think of the consequences of either being on the street, especially with the cold weather at the moment or you may have a roof over your head, sometimes there is only one choice."
Mr Stevens believes building more one-bedroom flats could help ease the problem.
"The average price is a hundred pounds a week to keep someone in a rented flat, prison is a lot more than that so I would imagine it would save the public purse quite a few pounds," he said.
Official figures show 830 one-bedroom properties were built in the year to March 2016, of an overall total of 6,900 new properties in Wales.
Marc, 50, who has been in and out of prison for the past 20 years for burglary offences, said he struggled to find accommodation each time he was released.
He said he would ask himself: "Where am I going to stay? Where am I going to live? Have I got somewhere where I can see my daughter."
"You're put out among the same sort of people doing the same sort of thing, and it's difficult, it's difficult to get away from it. It's like every man for himself, there's nothing."
Marc has now found stable accommodation with homeless charity Emmaus and said it had been life changing.
"You feel safe, you got hot food, you've got company of people in similar situations to yourself but all dealing with different issues. It's a constructive, helpful atmosphere," he said.
Tom Clarke, chief executive of Emmaus South Wales, agreed there was not enough support available.
"We do still see [people] homeless on the streets, so clearly they haven't got accommodation and haven't got provision," he said.
"I think the key is connecting people with the services they need. I don't delude myself that Emmaus can offer a one size fits all for everyone, we can't.
"But there must be other opportunities and given suitable encouragement I believe that can and should happen."
A Welsh Government spokesman said the national pathway for homeless services to children, young people and adults in the secure estate had prevented many people from losing their home whilst serving their prison sentence.
It added there were already significant demands for one-bedroom flats across the public and private sector and it was providing 20,000 new affordable homes in the next five years. There is a "chronic" need for more housing for prison leavers in Wales, according to a charity. 38264402 Officers searched properties in the Waterfront Park and Colonsay View areas of the city on Wednesday.
Detectives said three firearms, ammunition and a five-figure sum of money were recovered.
A 26-year-old man who was arrested and charged appeared at Edinburgh Sheriff Court on Thursday. A man has appeared in court after firearms, ammunition and ca

### Single-node Batch Inference

For single-node batch inference, the native `.predict()` method can be used

In [0]:
latest_model = mlflow.pyfunc.load_model(
  model_uri=f"models:/{model_name}/{current_model_version}"
)
latest_model

Device set to use cpu


mlflow.pyfunc.loaded_model:
  artifact_path: dbfs:/databricks/mlflow-tracking/4117165248792101/logged_models/m-89f313273fcd430da9666bad27d54e8c/artifacts
  flavor: mlflow.transformers
  run_id: 14ea559047da4983938766dd8aaebae4

In [0]:
from pprint import pprint


prod_data_sample_pdf = prod_data_df.limit(2).toPandas()
summaries_sample = latest_model.predict(prod_data_sample_pdf["document"])
[pprint(s+"\n") for s in summaries_sample]

Token indices sequence length is longer than the specified maximum sequence length for this model (775 > 512). Running this sequence through the model will result in indexing errors
Your max_length is set to 200, but your input_length is only 63. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=31)


('in prison, he says. "There\'s a desperate need for it, finding suitable '
 'accommodation for those leaving prison there is just a lack of it '
 'everywhere" prison Link Cymru, which helps people find accommodation after '
 'their release, said issues such as domestic violence were now considered . '
 'the charity says investment in housing would be cheaper than jailing '
 'homeless repeat offenders .\n')
('recovered three firearms, ammunition and a five-figure sum of money . a '
 '26-year-old man who was arrested appeared at Edinburgh Sheriff Court on '
 'Thursday .\n')


[None, None]

### Multinode Batch Inference
Below, we load the model using `mlflow.pyfunc.spark_udf`.  This returns the model as a Spark User Defined Function which can be applied efficiently to big data.  *Note that the deployment code is library-agnostic: it never references that the model is a Hugging Face pipeline.*  This simplified deployment is possible because MLflow logs environment metadata and "knows" how to load the model and run it.

In [0]:
# Grab `Champion` model (supposed to be latest production version)
prod_model_udf = mlflow.pyfunc.spark_udf(
    spark,
    model_uri=f"models:/{model_name}@champion",
    env_manager="local",
    result_type="string",
)

2025/11/15 22:51:01 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2025/11/15 22:51:02 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


In [0]:
dir(prod_model_udf)

['__annotations__',
 '__builtins__',
 '__call__',
 '__class__',
 '__closure__',
 '__code__',
 '__defaults__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__get__',
 '__getattribute__',
 '__getstate__',
 '__globals__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__kwdefaults__',
 '__le__',
 '__lt__',
 '__module__',
 '__name__',
 '__ne__',
 '__new__',
 '__qualname__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__wrapped__',
 '_unwrapped',
 'asNondeterministic',
 'deterministic',
 'evalType',
 'func',
 'metadata',
 'returnType']


When you load the model via MLflow above, you may see warnings about the Python environment.  It is very important to ensure that the environments for development, staging, and production match.
* For this demo notebook, everything is done within the same notebook environment, so we do not need to worry about libraries and versions.  However, in the Production the `env_manager` argument should be passed to the method while loading the saved MLflow model, to indicate what tooling to use to recreate the environment.
* To create a genuine production job, make sure to install the needed libraries.  MLflow saves these libraries and versions alongside the logged model; see the [MLflow docs on model storage](https://mlflow.org/docs/latest/models.html#storage-format) for more information.  While using Databricks for this course, you can also generate an example inference notebook which includes code for setting up the environment; see [the model inference docs](https://docs.databricks.com/machine-learning/manage-model-lifecycle/index.html#use-model-for-inference) for batch or streaming inference for more information.

In [0]:
# Run inference by appending a new column to the DataFrame

batch_inference_results_df = prod_data_df.withColumn("generated_summary", prod_model_udf("document"))
display(batch_inference_results_df)

document summary id generated_summary Prison Link Cymru had 1,099 referrals in 2015-16 and said some ex-offenders were living rough for up to a year before finding suitable accommodation.
Workers at the charity claim investment in housing would be cheaper than jailing homeless repeat offenders.
The Welsh Government said more people than ever were getting help to address housing problems.
Changes to the Housing Act in Wales, introduced in 2015, removed the right for prison leavers to be given priority for accommodation.
Prison Link Cymru, which helps people find accommodation after their release, said things were generally good for women because issues such as children or domestic violence were now considered.
However, the same could not be said for men, the charity said, because issues which often affect them, such as post traumatic stress disorder or drug dependency, were often viewed as less of a priority.
Andrew Stevens, who works in Welsh prisons trying to secure housing for prison leavers, said the need for accommodation was "chronic".
"There's a desperate need for it, finding suitable accommodation for those leaving prison there is just a lack of it everywhere," he said.
"It could take six months to a year, without a lot of help they could be on the streets for six months.
"When you think of the consequences of either being on the street, especially with the cold weather at the moment or you may have a roof over your head, sometimes there is only one choice."
Mr Stevens believes building more one-bedroom flats could help ease the problem.
"The average price is a hundred pounds a week to keep someone in a rented flat, prison is a lot more than that so I would imagine it would save the public purse quite a few pounds," he said.
Official figures show 830 one-bedroom properties were built in the year to March 2016, of an overall total of 6,900 new properties in Wales.
Marc, 50, who has been in and out of prison for the past 20 years for burglary offences, said he struggled to find accommodation each time he was released.
He said he would ask himself: "Where am I going to stay? Where am I going to live? Have I got somewhere where I can see my daughter."
"You're put out among the same sort of people doing the same sort of thing, and it's difficult, it's difficult to get away from it. It's like every man for himself, there's nothing."
Marc has now found stable accommodation with homeless charity Emmaus and said it had been life changing.
"You feel safe, you got hot food, you've got company of people in similar situations to yourself but all dealing with different issues. It's a constructive, helpful atmosphere," he said.
Tom Clarke, chief executive of Emmaus South Wales, agreed there was not enough support available.
"We do still see [people] homeless on the streets, so clearly they haven't got accommodation and haven't got provision," he said.
"I think the key is connecting people with the services they need. I don't delude myself that Emmaus can offer a one size fits all for everyone, we can't.
"But there must be other opportunities and given suitable encouragement I believe that can and should happen."
A Welsh Government spokesman said the national pathway for homeless services to children, young people and adults in the secure estate had prevented many people from losing their home whilst serving their prison sentence.
It added there were already significant demands for one-bedroom flats across the public and private sector and it was providing 20,000 new affordable homes in the next five years. There is a "chronic" need for more housing for prison leavers in Wales, according to a charity. 38264402 in prison, he says. "There's a desperate need for it, finding suitable accommodation for those leaving prison there is just a lack of it everywhere" prison Link Cymru, which helps people find accommodation after their release, said issues such as domestic violence were now considered . the charity says investment in housing would

In [0]:
prod_data_summaries_table_name = f"{DA.catalog_name}.{DA.schema_name}.m4_1_batch_inference"
batch_inference_results_df.write.mode("append").saveAsTable(prod_data_summaries_table_name)

## Batch Inference Using `ai_query()`

Another alternative & common method to run "batch-like" jobs using LLMs made available via the databricks foundation models API is to use the `ai_query()` **SQL** function [AWS](https://docs.databricks.com/en/sql/language-manual/functions/ai_query.html) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/ai_query)

In [0]:
%sql
CREATE OR REPLACE TABLE ai_query_inference AS (
  SELECT
  id
  ,ai_query(
    "databricks-meta-llama-3-3-70b-instruct",
    CONCAT("Based on the following document, provide a summary in less than 100 words. Document: ", document)
  ) as generated_summary
 FROM m4_1_prod_data LIMIT 10
)

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM ai_query_inference

id,generated_summary
38264402,"Here is a summary of the document in under 100 words: Prison Link Cymru, a charity that helps ex-offenders find accommodation, reports a ""chronic"" need for housing. Many ex-offenders, particularly men, are forced to live on the streets for up to a year before finding a place to live. The charity argues that investing in housing would be cheaper than jailing repeat offenders, and suggests building more one-bedroom flats as a solution. The Welsh Government claims to be providing support, but charities say more needs to be done to address the issue."
34227252,"Police searched properties in Edinburgh's Waterfront Park and Colonsay View areas, recovering three firearms, ammunition, and a large sum of money. A 26-year-old man was arrested, charged, and appeared in court on Thursday."
38537698,"Four Chicagoans, aged 18-24, were denied bail for allegedly kidnapping and torturing an 18-year-old white man with disabilities, forcing him to drink toilet water and making derogatory statements against white people. The incident was livestreamed on Facebook and has sparked outrage. The suspects face hate crime and kidnapping charges. An online fundraiser for the victim has raised $51,000."
36175342,"A 48-year-old former Arsenal goalkeeper has had a long tenure with West Brom, playing for the team and later serving as youth academy director and director of football. He played a key role in the team's two promotions to the Premier League in 2006 and 2012."
39070183,"Researchers found that a ""fasting-mimicking diet"" reversed symptoms of diabetes in animal experiments by regenerating cells in the pancreas. The diet, which involves 5 days of low-calorie eating followed by 25 days of normal eating, showed promise in treating both type 1 and type 2 diabetes. Experts say the findings are ""potentially very exciting"" but more research is needed to confirm the results in humans."
38899892,"Here is a summary of the document in under 100 words: Essilor, the world's largest lens manufacturer, and Luxottica, the leading frame manufacturer, are planning to merge. The combined company, EssilorLuxottica, would control a significant portion of the eyewear industry, raising concerns about reduced competition and higher prices. Industry experts warn that the merger could lead to a stranglehold on the supply of high-end glasses and squeeze out smaller competitors, ultimately affecting consumers. The European Commission is expected to review the merger to determine if it is in the public interest."
39339718,"Olympic silver medallist Wendy Houvenaghel has accused British Cycling of ""ageism"" and having ""zero regard"" for her welfare. She claims she was discarded from the team in 2012 due to her age and experienced a ""culture of fear"" and bullying. Her comments come after other high-profile cyclists criticized the organization's treatment of athletes, with British Cycling admitting to past cultural and governance failings and promising reforms."
34571446,"Kareem Badr and friends bought a struggling comedy club, The Hideout, in 2009 for $20,000. Despite no business experience, they turned it around and now employ 25 workers. The US comedy club industry is growing, with revenue expected to reach $344.6m by 2020, driven by a new golden age of stand-up comedy and a demand for live entertainment."
36892983,"Here is a summary of the document in under 100 words: Ofcom decided not to break up BT due to ""practical obstacles"" including complications with land deals and BT's £40bn pension scheme. The scheme has a £10bn deficit and is largely related to Openreach, which rivals want spun off. Instead, Ofcom will introduce a new board and articles of association to separate Openreach, which may lead to conflicts and potentially a court battle. BT's share price rose 3% on the news, but the company may still face a break-up if it cannot reach a compromise with Ofcom."
37732028,"Celtic manager Brendan Rodgers is looking forward to his team'


## Conclusion

In this demo, we developed a batch inference workflow using a small language model. First, we created a pipeline to summarize text. Then, we developed a model and registered it with Unity Catalog. We demonstrated how to track the model development process, query the model from the tracking server, and load the model back as a pipeline. We also showed how to manage model lifecycle with @aliases and concluded by showcasing two methods of batch inference: single-node and multi-node batch inference. Finally, we showed how to use `ai_query` for batch inference.

&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>